In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.tools import tool
from langchain_core.tools import create_retriever_tool

from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from langsmith import Client

from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

from platform import python_version

C:\Users\muham\AppData\Local\Temp\ipykernel_29348\3641853153.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
d:\projects\Langchain module tools and agents\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
wikipedia_tool = WikipediaQueryRun(api_wrapper = WikipediaAPIWrapper())

In [4]:
vectorstore = Chroma(persist_directory= "./intro-to-ds-lectures",
                     embedding_function= HuggingFaceEmbeddings())

retriever = vectorstore.as_retriever(search_type = 'mmr', 
                                     search_kwargs = {'k':3, 'lambda_mul':0.7})

retriever_tool = create_retriever_tool(retriever = retriever,
                                       name = "Introduction-to-Data-and-Data-Science-Course-Lectures",
                                       description = '''For any questions regarding the Introduction
                                       to Data and Data Science course, you must use this tool.''')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1867.05it/s]
C:\Users\muham\AppData\Local\Temp\ipykernel_29348\1897684529.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory= "./intro-to-ds-lectures",


In [5]:
@tool("Another Name")
def get_python_version() -> str:
    """Useful for questions regarding the version of Python currently used."""
    return python_version()

In [6]:
tools = [wikipedia_tool, retriever_tool, get_python_version]

In [7]:
chat = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [9]:
client = Client()
chat_prompt_template = client.pull_prompt(
    "hwchase17/openai-tools-agent", 
    dangerously_pull_public_prompt=True
)

print(type(chat_prompt_template))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [13]:
for tool in tools:
    tool.name = tool.name.replace(" ", "_")

print("Sanitized tool names:", [t.name for t in tools])

Sanitized tool names: ['wikipedia', 'Introduction-to-Data-and-Data-Science-Course-Lectures', 'Another_Name']


In [14]:
agent = create_tool_calling_agent(llm = chat,
                                  tools = tools,
                                  prompt = chat_prompt_template)

In [15]:
agent_executer = AgentExecutor(agent = agent,
                               tools = tools,
                               verbose = True,
                               return_intermediate_steps= True)

In [16]:
response = agent_executer.invoke({"input":"Could you tell me the version of Python I'm currently using?"})



> Entering new AgentExecutor chain...

Invoking: `Another_Name` with `{}`


3.13.0You’re running Python **3.13.0**.

> Finished chain.


In [17]:
response

{'input': "Could you tell me the version of Python I'm currently using?",
 'output': 'You’re running Python **3.13.0**.',
 'intermediate_steps': [(ToolAgentAction(tool='Another_Name', tool_input={}, log='\nInvoking: `Another_Name` with `{}`\n\n\n', message_log=[AIMessageChunk(content='', additional_kwargs={'reasoning_content': "We need to use the tool Another_Name. The function name is Another_Name. It takes no arguments. Let's call it.", 'tool_calls': [{'index': 0, 'id': 'fc_b078f276-cc29-4e9e-9288-fde8a6e3ec94', 'function': {'arguments': '{}', 'name': 'Another_Name'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq', 'finish_reason': 'tool_calls', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand'}, id='lc_run--01a0827a-07f9-7e12-876b-8cfb564c0305', tool_calls=[{'name': 'Another_Name', 'args': {}, 'id': 'fc_b078f276-cc29-4e9e-9288-fde8a6e3ec94', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

In [18]:
response = agent_executer.invoke({"input": """
I'm currently following the Introduction to Data and Data Science course.
Could you list the programming languages a data scientist should know?
Additionally, could you tell me who their creators are?
"""
})



> Entering new AgentExecutor chain...
Here’s a quick “cheat‑sheet” of the most common programming languages a data scientist should know, along with the people (or teams) who created them:

| Language | Typical Use in Data Science | Creator(s) | Year Created |
|----------|----------------------------|------------|--------------|
| **Python** | General‑purpose, data wrangling, machine learning, deep learning, visualization | **Guido van Rossum** | 1991 |
| **R** | Statistical analysis, graphics, reproducible research | **Ross Ihaka** & **Robert Gentleman** (University of Auckland) | 1993 |
| **SQL** | Relational database querying, data extraction | **Donald D. Chamberlin** & **Raymond F. Boyce** (IBM) | 1974 |
| **Julia** | High‑performance numerical computing, ML prototyping | **Jeff Bezanson**, **Stefan Karpinski**, **Viral B. Shah**, **Alan Edelman** | 2012 |
| **Scala** | Functional programming, Spark integration | **Martin Odersky** | 2004 |
| **Java** | Enterprise data pipelines

In [20]:
response

{'input': "\nI'm currently following the Introduction to Data and Data Science course.\nCould you list the programming languages a data scientist should know?\nAdditionally, could you tell me who their creators are?\n",
 'output': 'Here’s a quick “cheat‑sheet” of the most common programming languages a data scientist should know, along with the people (or teams) who created them:\n\n| Language | Typical Use in Data Science | Creator(s) | Year Created |\n|----------|----------------------------|------------|--------------|\n| **Python** | General‑purpose, data wrangling, machine learning, deep learning, visualization | **Guido van Rossum** | 1991 |\n| **R** | Statistical analysis, graphics, reproducible research | **Ross Ihaka** & **Robert Gentleman** (University of Auckland) | 1993 |\n| **SQL** | Relational database querying, data extraction | **Donald D. Chamberlin** & **Raymond F. Boyce** (IBM) | 1974 |\n| **Julia** | High‑performance numerical computing, ML prototyping | **Jeff Beza